# Tea Leaf Disease Classification — VGG16 Upgraded (Partial Fine-Tuning + Residual Connections)
CSC4093/DSC4213 Programming Assignment 02

**Upgrades over baseline VGG16:**
- Unfrozen last 2 conv blocks (features[17:]) for fine-tuning
- Custom classifier with residual (skip) connection
- Dropout regularization at multiple points
- Batch Normalization added


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np
import os, time, copy
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# ── Configuration ─────────────────────────────────────────
DATA_DIR    = './dataset'   # <-- Update to your dataset path
NUM_CLASSES = 3
BATCH_SIZE  = 16
NUM_EPOCHS  = 20            # More epochs for fine-tuning
LR          = 0.0005        # Lower LR for fine-tuning unfrozen layers
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')


In [ ]:
# ── Data Transforms ───────────────────────────────────────
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ── Dataset Split: 70% train / 15% val / 15% test ─────────
full_dataset = datasets.ImageFolder(DATA_DIR, transform=train_transforms)
class_names  = full_dataset.classes
print(f'Classes: {class_names}  |  Total images: {len(full_dataset)}')

n_total = len(full_dataset)
n_train = int(0.70 * n_total)
n_val   = int(0.15 * n_total)
n_test  = n_total - n_train - n_val

train_set, val_set, test_set = random_split(
    full_dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)
val_set.dataset         = copy.deepcopy(full_dataset)
val_set.dataset.transform  = val_transforms
test_set.dataset        = copy.deepcopy(full_dataset)
test_set.dataset.transform = val_transforms

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f'Train: {n_train} | Val: {n_val} | Test: {n_test}')


In [ ]:
# ── Residual Classifier Block ──────────────────────────────
# This adds a skip connection inside the custom classifier head.
# If input and output dims match, the input is added directly (identity).
# Otherwise a 1x1 linear projection aligns the dimensions.
class ResidualBlock(nn.Module):
    def __init__(self, in_features, out_features, dropout=0.4):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(in_features, out_features),
            nn.BatchNorm1d(out_features),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(out_features, out_features),
            nn.BatchNorm1d(out_features),
        )
        # Projection shortcut if dims differ
        self.shortcut = (
            nn.Linear(in_features, out_features, bias=False)
            if in_features != out_features else nn.Identity()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.block(x) + self.shortcut(x))


# ── Upgraded VGG16 Model ───────────────────────────────────
class VGG16Upgraded(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        base = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

        # Freeze early layers (features[0:17] = blocks 1-3)
        for i, layer in enumerate(base.features):
            if i < 17:
                for param in layer.parameters():
                    param.requires_grad = False
            # features[17:] = blocks 4 & 5 — unfrozen for fine-tuning

        self.features   = base.features
        self.avgpool    = base.avgpool        # Adaptive avg pool → 7x7
        self.flatten    = nn.Flatten()

        # Custom classifier with residual connections
        self.pre        = nn.Sequential(
            nn.Linear(25088, 2048),           # 512*7*7 = 25088
            nn.BatchNorm1d(2048),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
        )
        self.res1       = ResidualBlock(2048, 1024, dropout=0.4)
        self.res2       = ResidualBlock(1024, 512,  dropout=0.3)
        self.classifier = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = self.flatten(x)
        x = self.pre(x)
        x = self.res1(x)
        x = self.res2(x)
        return self.classifier(x)


model_vgg = VGG16Upgraded(NUM_CLASSES).to(DEVICE)

trainable = sum(p.numel() for p in model_vgg.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model_vgg.parameters())
print(f'Trainable: {trainable:,} / Total: {total:,} parameters')

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)  # Label smoothing prevents overconfidence

# Use different LRs: smaller for unfrozen conv layers, larger for new classifier
conv_params  = [p for n, p in model_vgg.named_parameters() if 'features' in n and p.requires_grad]
head_params  = [p for n, p in model_vgg.named_parameters() if 'features' not in n and p.requires_grad]

optimizer = optim.Adam([
    {'params': conv_params, 'lr': LR * 0.1},   # 10x smaller for fine-tuned conv layers
    {'params': head_params, 'lr': LR},
], weight_decay=1e-4)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
print('Model built successfully!')


In [ ]:
# ── Training Loop ─────────────────────────────────────────
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs):
    history  = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
    best_acc = 0.0
    best_wts = copy.deepcopy(model.state_dict())
    t0 = time.time()

    for epoch in range(num_epochs):
        for phase, loader in [('train', train_loader), ('val', val_loader)]:
            model.train() if phase == 'train' else model.eval()
            running_loss, running_corrects = 0.0, 0

            for inputs, labels in loader:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    loss    = criterion(outputs, labels)
                    preds   = outputs.argmax(dim=1)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                running_loss     += loss.item() * inputs.size(0)
                running_corrects += (preds == labels).sum().item()

            epoch_loss = running_loss / len(loader.dataset)
            epoch_acc  = running_corrects / len(loader.dataset)
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc)

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_wts = copy.deepcopy(model.state_dict())

        scheduler.step()
        print(f'Epoch [{epoch+1:02d}/{num_epochs}]  '
              f'Train Loss: {history["train_loss"][-1]:.4f}  Acc: {history["train_acc"][-1]:.4f}  |  '
              f'Val Loss: {history["val_loss"][-1]:.4f}  Acc: {history["val_acc"][-1]:.4f}')

    elapsed = time.time() - t0
    print(f'\nDone in {elapsed//60:.0f}m {elapsed%60:.0f}s | Best Val Acc: {best_acc:.4f}')
    model.load_state_dict(best_wts)
    return model, history

model_vgg, history_vgg = train_model(
    model_vgg, train_loader, val_loader,
    criterion, optimizer, scheduler, NUM_EPOCHS
)
torch.save(model_vgg.state_dict(), 'vgg16_upgraded_tea_disease.pth')


In [ ]:
# ── Training Curves ───────────────────────────────────────
epochs_range = range(1, NUM_EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_range, history_vgg['train_acc'], 'b-o', label='Train')
axes[0].plot(epochs_range, history_vgg['val_acc'],   'r-o', label='Validation')
axes[0].set_title('VGG16 Upgraded — Accuracy')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs_range, history_vgg['train_loss'], 'b-o', label='Train')
axes[1].plot(epochs_range, history_vgg['val_loss'],   'r-o', label='Validation')
axes[1].set_title('VGG16 Upgraded — Loss')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.savefig('vgg16_upgraded_training_curves.png', dpi=150)
plt.show()


In [ ]:
# ── Test Set Evaluation ───────────────────────────────────
model_vgg.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model_vgg(inputs.to(DEVICE))
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

print('\n── Classification Report ──')
print(classification_report(all_labels, all_preds, target_names=class_names))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('VGG16 Upgraded — Confusion Matrix')
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('vgg16_upgraded_confusion_matrix.png', dpi=150)
plt.show()
